In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

In [ ]:
X_train = np.array([[0,0],[0,1],[1,0],[1,1]], "float32")   # input X

y = np.array([[0],[1],[1],[0]], "float32")    # XOR
#y = np.array([[0],[0],[0],[1]], "float32")    # AND
#y = np.array([[0],[1],[1],[1]], "float32")    # OR
#y = np.array([[1],[1],[1],[0]], "float32")    # random

y_train = y

# Convert labels to one-hot encoding
y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=2)

In [ ]:
# Define the neural network architecture
model = tf.keras.Sequential([
    tf.keras.layers.Dense(2, input_dim=2, activation='softmax')
])

model = Sequential()
model.add(Dense(units = 20, input_dim=2, activation = 'relu'))
model.add(Dense(units = 10, activation = 'relu'))
model.add(Dense(units = 2, activation = 'softmax'))                  # output layer

'''
model.add(Dense(units = 2, input_dim=2, activation = 'relu'))
model.add(Dense(units = 2, activation = 'softmax'))                  # output layer
'''

"\nmodel.add(Dense(units = 2, input_dim=2, activation = 'relu'))\nmodel.add(Dense(units = 2, activation = 'softmax'))                  # output layer\n"

In [ ]:
# Custom cross-entropy loss function for 2-class classification
def custom_categorical_crossentropy(y_true, y_pred):
    epsilon = 1e-15  # Small constant to avoid log(0)
    y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
    return -tf.reduce_mean(y_true * tf.math.log(y_pred))


# **Still use "fit()" but use it with ONE epoch at a time using our own for loop**

In [ ]:
sgd = SGD(learning_rate=0.01)
model.compile(optimizer=sgd, loss=custom_categorical_crossentropy, metrics=['accuracy'])
epochs = 1000
for epoch in range(epochs):
    model.fit(X_train, y_train_one_hot, epochs=1, batch_size=len(X_train), verbose=0)

    # Evaluate the model after each epoch
    loss, accuracy = model.evaluate(X_train, y_train_one_hot, verbose=0)
    if epoch % 100 == 0:
      print(f'Epoch {epoch + 1}/{epochs}, Loss: {loss}, Accuracy: {accuracy}')


# **Does NOT use "fit()". Using our own for loop with our own gradient update.**

In [ ]:
# Custom training loop..... ***** BUT, NO handle of batch size
learning_rate = 0.01
epochs = 1000

optimizer = tf.keras.optimizers.SGD(learning_rate)

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        # Forward pass
        y_pred = model(X_train)
        # Loss value
        loss = custom_categorical_crossentropy(y_train_one_hot, y_pred)

    # Get gradients of loss wrt the weights.
    gradients = tape.gradient(loss, model.trainable_variables)
    # Update the weights of the model.
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    if epoch % 100 == 0:
        print(f'Epoch {epoch}/{epochs}, Loss: {loss.numpy()}')

Epoch 0/1000, Loss: 0.35883742570877075
Epoch 100/1000, Loss: 0.3300095200538635
Epoch 200/1000, Loss: 0.3117433786392212
Epoch 300/1000, Loss: 0.29458773136138916
Epoch 400/1000, Loss: 0.2786220908164978
Epoch 500/1000, Loss: 0.2639193534851074
Epoch 600/1000, Loss: 0.250148743391037
Epoch 700/1000, Loss: 0.23641160130500793
Epoch 800/1000, Loss: 0.22273513674736023
Epoch 900/1000, Loss: 0.20908033847808838


In [ ]:
# Evaluate the trained model
predictions = model(X_train).numpy()
predicted_classes = np.argmax(predictions, axis=1)
print(predictions)
print("\nPredicted Classes:")
print(predicted_classes)

[[0.56080914 0.4391909 ]
 [0.17533697 0.824663  ]
 [0.406378   0.593622  ]
 [0.75994045 0.24005957]]

Predicted Classes:
[0 1 1 0]


In [ ]:
print(y_train)
print(np.equal(predicted_classes, y_train.T))
# Compare predicted classes with actual labels
accuracy = np.mean(np.equal(predicted_classes, y_train.T))
print("\nAccuracy:", accuracy)


[[0.]
 [1.]
 [1.]
 [0.]]
[[ True  True  True  True]]

Accuracy: 1.0
